# 00 · Build

**Goal:** build everything the matching service loads, from the time-entry file: email fields filled in, clues for every entry and matter pair, both models, and the cutoffs for each preset.

Each step is a function in `src/`, the same ones a firm would use to retrain on its own history (`train.train`). This notebook runs them in order, checks the data along the way, and saves the files the app reads.

| Step | Function | Saves to `results/` |
|---|---|---|
| Fill in email fields | `parse_email_entries` (shared with extraction) | `00_entries.parquet` |
| Clues for every pair | `build_training_pairs` | `02_pairs.parquet`, `02_people.parquet`, `02_config.json` |
| Both models and the cutoffs | `fit` | `04_model.joblib`, `04_config.json`, `04_train_scores.parquet` |

Only TRAIN entries are learned from. VAL, TEST, and the holdout file are graded in `02_evaluation`.

The first run downloads the sentence embedding model (about 90 MB). The grid search in section 5 is the slow part.

## 1. Setup

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from sentence_transformers import SentenceTransformer

sys.path.insert(0, "../src")
sys.path.insert(0, "../../extraction/src")
from classifier import route_actions, top_k_accuracy
from evaluate import parse_metrics
from fuzzy import client_name_table, matter_name_table
from pipeline import decide, load_entries, load_reference
from rules import answer_key, run_rules, score, summarize
from text_parser import parse_email_entries
from train import GRIDS, SETTINGS, build_training_pairs, fit, save

In [2]:
DATA_PATH = Path("../data/synthetic_time_entries_v3.xlsx")
SHORTHAND_PATH = Path("../data/shorthand_bank.csv")   # billing shorthand, never read as client initials
RESULTS_DIR = Path("../results")

ENTRY_COLS = {"Time ID": "entry_id", "TK ID": "tk_id", "Source Format": "source_format", "Date": "work_date",
              "CMID": "cmid", "Client": "client", "Matter": "matter", "Narrative": "narrative", "Hours": "hours"}
MATCH_COLS = ["entry_id", "tk_id", "source_format", "work_date", "cmid", "client", "matter", "narrative"]
HISTORY_START, HISTORY_END = pd.Timestamp("2026-04-01"), pd.Timestamp("2026-06-30")   # the dates TK Matter History covers
SKIP_CLIENTS = ["Firm"]   # an ordinary word, so "firm meeting" doesn't name a client
STRICT, DEFAULT = 0.98, 0.97   # share of automatic decisions that have to be right (balanced: section 5)

## 2. Load data

- **Entries** and the **answer key** (`KEY:` columns) come from the same file. The answer key is kept apart and only used for labels and checks.
- **Two fixed choices** come from earlier analysis and are read from `results/`: the fuzzy scorers for typed client and matter names (`01_scorer_choice.json`) and the hand-written rules that serve as the baseline (`03_config.json`).

In [3]:
sheets = pd.read_excel(DATA_PATH, sheet_name=None, dtype=str)
raw, matter_master = sheets["Time Entries"], sheets["Matter Master"]

entries = raw[list(ENTRY_COLS)].rename(columns=ENTRY_COLS)
answers = (raw[["Time ID"] + [c for c in raw.columns if c.startswith("KEY: ")]]
           .rename(columns=lambda c: "entry_id" if c == "Time ID" else c.replace("KEY: ", "key_").lower().replace(" ", "_"))
           .rename(columns={"key_split": "split"}))
acronym_skip = pd.read_csv(SHORTHAND_PATH)["shorthand"].tolist()
scorer_choice = json.loads((RESULTS_DIR / "01_scorer_choice.json").read_text())
rules_config = json.loads((RESULTS_DIR / "03_config.json").read_text())

assert entries["entry_id"].is_unique and entries["entry_id"].tolist() == answers["entry_id"].tolist()
assert not any(c.startswith("KEY") for c in entries.columns)
assert set(answers["split"]) == {"TRAIN", "VAL", "TEST"}
assert pd.to_datetime(entries["work_date"]).min() > HISTORY_END
print(f"{len(entries):,} entries | {matter_master['CMID'].nunique()} matters | {matter_master['Client'].nunique()} clients")
pd.crosstab(entries["source_format"], answers["split"], margins=True)

4,000 entries | 103 matters | 35 clients


split,TEST,TRAIN,VAL,All
source_format,,,,
EMAIL,49,273,67,389
HANDWRITTEN,124,554,109,787
PDF,107,458,99,664
TYPED,317,1517,326,2160
All,597,2802,601,4000


## 3. Fill in email fields

Typed sheets, PDFs, and handwritten sheets come with CMID, client, and matter fields, but emails come with only a body. `parse_email_entries` reads the CMID, client, matter, and hours out of each email body, using the firm's client and matter lists, so emails go into matching like every other entry. Other formats pass through unchanged.

The check uses TRAIN and VAL emails only.

In [4]:
client_names, matter_names = client_name_table(matter_master, acronym_skip), matter_name_table(matter_master)
parsed = parse_email_entries(entries, client_names, matter_names, skip=SKIP_CLIENTS)
is_email = parsed["source_format"] == "EMAIL"

assert parsed.index.equals(entries.index)
assert parsed.loc[~is_email, MATCH_COLS].equals(entries.loc[~is_email, MATCH_COLS])

tuned = parsed[is_email & answers["split"].ne("TEST")]
filled = pd.DataFrame({"before": entries.loc[is_email, ["cmid", "client", "matter"]].notna().mean(),
                       "after": parsed.loc[is_email, ["cmid", "client", "matter"]].notna().mean()})
print(f"{is_email.sum()} emails | entries found per email: {parsed.loc[is_email, 'n_parsed'].value_counts().to_dict()}")
print(pd.Series(parse_metrics(tuned, answers, client_names)).round(3).to_dict())
filled.round(2)

389 emails | entries found per email: {1: 389}
{'emails': 340.0, 'hours_found': 1.0, 'client_filled': 0.756, 'right_client': 0.72, 'wrong_client': 0.023}


,before,after
cmid,0.0,0.05
client,0.0,0.76
matter,0.0,0.29


**Takeaways**

- Hours are found in every email, and the client is filled in for about 3 in 4 (76% of TRAIN and VAL emails). A filled-in client is almost always right: 72% of emails get the right client and 2.3% a wrong one.
- An email with no client found still goes on to matching, which falls back on the narrative and billing history, like a typed entry with a blank client.

## 4. Clues for every pair

Every entry is paired with every matter, so the right matter is always a candidate, and each pair is described by clues:

| Group | What it says about the pair |
|---|---|
| Typed fields | which fields were filled in, and the source format |
| CMID | the typed CMID is this matter, a misread of it, or at least the same client |
| Names | fuzzy scores of the typed client and matter against this matter |
| Narrative | key terms from the matter's name, task cues for its type of work, the client named |
| Dates | the matter wasn't open on the work date |
| History | how much this timekeeper billed this matter, per week and as a share of their time |
| Gaps | each clue minus the best value among the entry's candidates |
| Past narratives | how much the narrative reads like past entries on this matter (sentence embeddings) |
| People | people named in the narrative who were billed with this matter's client before |

**No peeking:** TRAIN entries stand in for billing history. TRAIN is split into 5 folds, and a TRAIN entry only uses history from the other folds, the same way the models are cross-validated.

In [5]:
encoder = SentenceTransformer(SETTINGS["embedding_model"])
pairs, features, info = build_training_pairs(parsed[MATCH_COLS + ["hours", "parsed_hours"]], answers, sheets, encoder, scorer_choice, acronym_skip,
                                             HISTORY_START, HISTORY_END)
feature_cols = [c for cols in features.values() for c in cols]
labels = answer_key(pairs)

per_entry = pairs.groupby("entry_id")
expected = labels.set_index("entry_id")["key_expected_action"].map({"MULTI_MATTER": 2, "NO_MATCH": 0}).fillna(1)
assert per_entry.size().eq(matter_master["CMID"].nunique()).all()
assert pairs[feature_cols].notna().all().all() and pairs["hours"].notna().all()
assert per_entry["is_true_matter"].sum().eq(expected.reindex(per_entry.size().index)).all()
assert pairs[pairs["split"] == "TRAIN"].groupby("entry_id")["fold"].nunique().eq(1).all()
print(f"{len(pairs):,} pairs | {len(feature_cols)} clues | {len(info['people'])} people in billing history")
pd.Series({group: ", ".join(cols) for group, cols in features.items()}, name="clues").to_frame()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

412,000 pairs | 48 clues | 67 people in billing history


,clues
typed_fields,"has_cmid, has_client, has_matter, client_is_di..."
cmid,"cmid_valid, cmid_client_known, is_typed_cmid, ..."
names,"client_ratio, client_token_sort_ratio, client_..."
narrative,"keyterm_score, keyterm_hits, own_keyterm_hits,..."
dates,"not_open, opened_after_history"
history,"tk_known, in_history, history_per_week, histor..."
gaps,"client_ratio_gap, matter_token_sort_ratio_gap,..."
past_narratives,"past_narrative_sim, past_narrative_sim_gap"
people,"person_hits, person_conflict"


## 5. Both models and the cutoffs

`fit` runs three steps on TRAIN, all tuned on the same 5 folds:
1. **Pair model:** LightGBM scores every pair with the probability that it's the right matter. Its settings are picked by log loss.
2. **Action model:** a second LightGBM looks at each entry's top candidates and clues and gives the chance the top pick is safe to accept, needs a person, covers two matters, or is a new client. A narrative-only text model feeds it a score for wording about other work, and the entry's hours and narrative length help it spot entries that cover more than one matter (a long, multi-task entry is how block billing usually shows up). Missing hours get the TRAIN median.
3. **Cutoffs:** for each preset's target, the cutoffs that match the most entries with no person while still reaching the target in 90% of timekeeper resamples. Correcting a typed CMID has its own bar (`correct_min`), since the corrections on their own have to reach the target.

The presets are `strict` (98%), `default` (97%), and `balanced`, which only has to be as accurate as the hand-written rules on TRAIN.

In [6]:
train_pairs = pairs[pairs["split"] == "TRAIN"].reset_index(drop=True)
rules_train = summarize(score(run_rules(train_pairs, rules_config["rules"]), labels))
TARGETS = {"strict": STRICT, "default": DEFAULT, "balanced": rules_train["decided_precision"]}
print("Targets:", {name: round(t, 3) for name, t in TARGETS.items()})

trained = fit(pairs, features, info, SETTINGS | {"targets": TARGETS}, GRIDS)
print(f"Pair model: {trained['pair_search'].best_params_} | log loss {-trained['pair_search'].best_score_:.4f} | "
      f"top-1 {top_k_accuracy(train_pairs, trained['cv_probs'], k=1):.3f} | top-3 {top_k_accuracy(train_pairs, trained['cv_probs'], k=3):.3f}")
print(f"Text model: {trained['text_search'].best_params_} | action model: {trained['action_search'].best_params_} | "
      f"log loss {-trained['action_search'].best_score_:.3f}")

Targets: {'strict': 0.98, 'default': 0.97, 'balanced': 0.861}
Pair model: {'min_child_samples': 100, 'n_estimators': 200, 'num_leaves': 31} | log loss 0.0088 | top-1 0.924 | top-3 0.975
Text model: {'logisticregression__C': 10} | action model: {'min_child_samples': 20, 'n_estimators': 100, 'num_leaves': 7} | log loss 0.252


### The presets on TRAIN

Every TRAIN entry here is scored by models that never saw it, so these are fair numbers for TRAIN, and `02_evaluation` checks them on the other sets.

In [7]:
COLS = ["auto_share", "auto_precision", "decided_precision", "silent_errors", "review_share", "flag_share", "multi_caught"]
rows = {}
for name, cfg in trained["model_config"]["routing"].items():
    s = summarize(score(route_actions(trained["entries"], trained["action_cv"], cfg), labels))
    rows[name] = {"target": TARGETS[name], "auto_min": cfg["auto_min"], "correct_min": cfg.get("correct_min", "off"),
                  **{k: s[k] for k in COLS}}
presets = pd.DataFrame(rows).T
presets["reached_target"] = presets["decided_precision"] >= presets["target"]
presets.round(3)

,target,auto_min,correct_min,auto_share,auto_precision,decided_precision,silent_errors,review_share,flag_share,multi_caught,reached_target
strict,0.98,0.95,off,0.564954,0.986734,0.985465,21,0.211991,0.093505,0.850575,True
default,0.97,0.9,0.97,0.645967,0.98011,0.979456,36.0,0.160243,0.06424,0.850575,True
balanced,0.860995,0.5,0.8,0.796931,0.944021,0.936179,125.0,0.021413,0.037473,0.844828,True


**Takeaways**


- All three presets reach their targets on TRAIN, scored by models that never saw each entry.
- **Default (97%):** 65% of entries matched with no person, 98.0% of them right, 36 silent errors in 2,802 entries, and 16% sent to review.
- **Strict (98%):** 56% matched, 98.7% right, 21 silent errors, and 21% to review. It never overrules a typed CMID: no correction bar reached 98% on its own, so a CMID that disagrees with the model is flagged instead.
- **Balanced (the rules' 86%):** 80% matched, 94.4% right, and 125 silent errors, 3.5 times the default's.
- **Multi-matter entries:** 85% are caught at every preset, since all three share one multi-matter cutoff (0.15). Hours and narrative length bring step 2's log loss to 0.252 (0.260 without them).

## 6. Save

- `00_entries.parquet`: every entry with email fields filled in (`parsed_hours` and `n_parsed` kept for reference)
- `02_pairs.parquet`, `02_people.parquet`, `02_config.json`: the pairs table, who was billed with which client, and the clue settings
- `04_model.joblib`, `04_config.json`: both models, the text model, and each preset's cutoffs
- `04_train_scores.parquet`: the TRAIN scores `repick_cutoffs` uses to set a new target without retraining

The check loads the saved files the way the app does and decides a few VAL entries.

In [8]:
parsed.drop(columns=["hours"]).to_parquet(RESULTS_DIR / "00_entries.parquet", index=False)
save(trained, RESULTS_DIR)

ref = load_reference(DATA_PATH, RESULTS_DIR, encoder, "default")
check_entries, check_answers = load_entries(DATA_PATH, ref, SKIP_CLIENTS)
sample = check_entries[check_answers["split"].eq("VAL").to_numpy()].head(20)
_, _, sample_decisions = decide(sample, ref)
assert len(sample_decisions) == len(sample) and sample_decisions["reason_short"].notna().all()
print("Saved:", ", ".join(sorted(p.name for p in RESULTS_DIR.glob("0[024]_*"))))
sample_decisions["action"].value_counts().to_frame("20 VAL entries")

Saved: 00_entries.parquet, 02_config.json, 02_pairs.parquet, 02_people.parquet, 04_config.json, 04_model.joblib, 04_train_scores.parquet


,20 VAL entries
action,
AUTO_MATCH,9
NEEDS_REVIEW,4
MULTI_MATTER,3
FLAG_MISMATCH,3
NO_MATCH,1


## 7. Findings

- **Step 1 ranks well:** the right matter is first for 92.4% of TRAIN entries and in the top 3 for 97.5%, out of fold.
- **The default preset matches about 2 in 3 entries with no person at 98% right on TRAIN.** Strict sends about 5 more entries per 100 to review for 15 fewer silent errors (36 to 21).
- **Hours and narrative length help step 2 spot multi-matter entries:** 85% are caught on TRAIN. The synthetic multi-matter entries are long by design, so a real firm would likely see a smaller gain and more false alarms from long entries on one matter.
- **Everything is picked on TRAIN alone** (models, cutoffs, and the correction bar), so `02_evaluation` checks it on VAL, TEST, and the holdout file with nothing tuned on them.
- **The saved files work in the app's path:** `load_reference` and `decide` run on 20 VAL entries and give every decision a reason.